# Dauphin Island spring migrants: answering the research questions

Input: `data/zerofilled.parquet` from **`dauphin_dataset.ipynb`**, with one row per checklist × target species.

Order of the analysis (each step uses the one before it):

| Section | Question | Main outputs |
|---|---|---|
| **RQ3** | Which covariates matter? How do season, time and place bias the data? | Effort-over-time plots, covariate importance tables |
| **RQ1** | How are the migrant populations changing? | Raw vs. adjusted trend plots, % change per year with 95% CIs, community trend |
| **RQ2** | How do we know the analysis is reliable? | Effort-tracking check, holdout AUC/Brier, permutation test, sensitivity table, external comparison |
| **RQ4** | Does a northern site support the trends? | Cross-site trend comparison |

All tables and figures are saved to `results/`.

In [ ]:
# Setup: pip install -r requirements.txt  (see README)
import os, warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import GroupKFold, train_test_split

warnings.filterwarnings("ignore")   # silences convergence/separation chatter from rare species
pd.set_option("display.precision", 3)

load_dotenv()   # reads machine-specific paths from .env (git-ignored); see .env.example

## Settings

In [ ]:
# Paths come from .env (see .env.example); the defaults below are used if a variable is unset.
DATA_DIR        = os.getenv("DATA_DIR", "data")
DATA            = os.path.join(DATA_DIR, "zerofilled.parquet")
NORTH_DATA      = os.path.join(os.getenv("NORTH_DATA_DIR", "data_north"), "zerofilled.parquet")   # RQ4: northern site (optional)
EXTERNAL_TRENDS = os.getenv("EXTERNAL_TRENDS_CSV", "external_trends.csv")   # RQ2: BBS / Status & Trends estimates (optional)
WEATHER_CSV     = os.getenv("WEATHER_CSV", "weather.csv")                   # RQ1: daily weather (optional)
RESULTS         = os.getenv("RESULTS_DIR", "results")

CONTROL_SPECIES = ["Northern Cardinal", "Carolina Wren"]   # residents, used as a check in RQ2

N_BOOT        = 100   # bootstrap runs for confidence intervals (200+ for the final report)
N_PERM        = 100   # permutation-test runs
HOLDOUT_YEARS = 3     # most recent years held out for testing
SEED          = 42

# "Standard checklist" used to compare years on equal footing
STD_START_HOUR   = 7      # 7 AM
STD_DURATION_MIN = 60     # 1 hour
STD_DISTANCE_KM  = 1.0
STD_OBSERVERS    = 1
STD_TRAVELING    = 1      # traveling count

os.makedirs(RESULTS, exist_ok=True)
rng = np.random.default_rng(SEED)

## Load and reshape

Reshape the long table into one row per checklist, with one 0/1 column per species. Then scale the covariates so the models fit cleanly:

- `doy`: day of year, centered on about April 20 (in units of 20 days)
- `hr`: start hour, centered on 9 AM (in units of 3 hours)
- `log_dur`: log of duration in minutes
- `traveling`: 1 for traveling counts, 0 for stationary
- `other_species`: species on the checklist **other than** the one being modeled, a proxy for observer skill (computed per species so it doesn't include the response)

In [ ]:
zf = pd.read_parquet(DATA)
SPECIES  = sorted(zf["common_name"].unique())
MIGRANTS = [s for s in SPECIES if s not in CONTROL_SPECIES]

CL_COLS = ["checklist_id", "obs_date", "year", "day_of_year", "start_hour", "week",
           "cell_x", "cell_y", "locality_id", "observer_id", "protocol",
           "duration_min", "distance_km", "n_observers", "n_species"]

def make_wide(zf):
    cl = zf[CL_COLS].drop_duplicates("checklist_id").set_index("checklist_id")
    det = zf.pivot_table(index="checklist_id", columns="common_name", values="detected", aggfunc="max")
    w = cl.join(det)
    w["year_c"]    = w["year"] - w["year"].median()
    w["doy"]       = (w["day_of_year"] - 110) / 20
    w["hr"]        = (w["start_hour"] - 9) / 3
    w["log_dur"]   = np.log(w["duration_min"].clip(lower=1))
    w["traveling"] = (w["protocol"] == "P22").astype(int)
    w["cell"]      = w["cell_x"].astype(str) + "_" + w["cell_y"].astype(str)
    return w

wide = make_wide(zf)
YEARS = np.sort(wide["year"].unique())
print(f"{len(wide):,} checklists, {len(SPECIES)} species, years {YEARS.min()}-{YEARS.max()}")
print("Migrants:", MIGRANTS)
print("Controls:", [s for s in SPECIES if s in CONTROL_SPECIES])

### Shared model helpers

Every model is a **logistic regression** (GLM, binomial family) of detection on the effort covariates:

```
detected ~ day of year (+ squared) + start hour (+ squared) + log(duration)
           + distance + observers + traveling + log(1 + other species)
           + year
```

Year enters either as a **number** (one slope, giving the trend) or as a **category** (one estimate per year, giving the yearly index).
To compare years fairly, each model predicts detection probability for one **standard checklist** at the species' median migration date.

In [ ]:
COV_TERMS = {   # covariate groups (RQ3 tests each group by dropping it)
    "day of year":        "doy + I(doy**2)",
    "start time":         "hr + I(hr**2)",
    "duration":           "log_dur",
    "distance":           "distance_km",
    "observers":          "n_observers",
    "protocol":           "traveling",
    "checklist richness": "np.log1p(other_species)",
}
COVARS = " + ".join(COV_TERMS.values())


def species_df(w, sp):
    """Checklist table with `detected` = this species and `other_species` excluding it."""
    d = w.copy()
    d["detected"] = d[sp]
    d["other_species"] = d["n_species"] - d["detected"]
    return d


def fit_glm(d, rhs):
    try:
        return smf.glm(f"detected ~ {rhs}", data=d, family=sm.families.Binomial()).fit()
    except Exception:
        return None


def std_frame(d, years):
    """Standard checklist, one row per year, at this species' median detection date."""
    det = d[d["detected"] == 1]
    doy = det["doy"].median() if len(det) else d["doy"].median()
    return pd.DataFrame({
        "year": years,
        "year_c": years - (d["year"] - d["year_c"]).iloc[0],   # same centering as the data
        "doy": doy,
        "hr": (STD_START_HOUR - 9) / 3,
        "log_dur": np.log(STD_DURATION_MIN),
        "distance_km": STD_DISTANCE_KM,
        "n_observers": STD_OBSERVERS,
        "traveling": STD_TRAVELING,
        "other_species": d["other_species"].median(),
    })


def annual_trend(d):
    """% change per year in detection probability on the standard checklist."""
    fit = fit_glm(d, f"year_c + {COVARS}")
    if fit is None:
        return np.nan
    y0, y1 = d["year"].min(), d["year"].max()
    p = fit.predict(std_frame(d, np.array([y0, y1])))
    return ((p.iloc[1] / p.iloc[0]) ** (1 / (y1 - y0)) - 1) * 100


def yearly_index(d):
    """Detection probability on the standard checklist, estimated separately for each year."""
    fit = fit_glm(d, f"C(year) + {COVARS}")
    years = np.sort(d["year"].unique())
    if fit is None:
        return pd.Series(np.nan, index=years)
    return pd.Series(fit.predict(std_frame(d, years)).values, index=years)


def spatial_subsample(w, seed):
    """One checklist per ~1 km grid cell per week per year."""
    return w.groupby(["year", "week", "cell_x", "cell_y"], group_keys=False).sample(n=1, random_state=seed)


def trend_table(w, species, n_boot=N_BOOT, label=""):
    """Trend on all data + bootstrap CI (spatial subsample, then resample checklists within year)."""
    rows = []
    boots = {sp: [] for sp in species}
    for b in range(n_boot):
        sub = spatial_subsample(w, seed=SEED + b)
        sub = sub.groupby("year", group_keys=False).sample(frac=1, replace=True, random_state=SEED + b)
        for sp in species:
            boots[sp].append(annual_trend(species_df(sub, sp)))
    for sp in species:
        bs = np.array(boots[sp], dtype=float)
        bs = bs[np.isfinite(bs)]
        rows.append({
            "species": sp,
            "trend_%/yr": annual_trend(species_df(w, sp)),
            "ci_low": np.percentile(bs, 2.5) if len(bs) else np.nan,
            "ci_high": np.percentile(bs, 97.5) if len(bs) else np.nan,
            "detections": int(w[sp].sum()),
        })
    t = pd.DataFrame(rows).set_index("species")
    t["direction"] = np.select([t["ci_low"] > 0, t["ci_high"] < 0], ["increase", "decrease"], "no clear change")
    return t

---
# RQ3: Which covariates matter, and how do season, time and place bias the data?

## RQ3a. How birder effort changed over time

If the number of checklists, their length or birders' skill changed over the years, raw sighting totals will show "trends" that are really about birders. These plots show how big that problem is.

In [ ]:
effort = wide.groupby("year").agg(
    checklists          = ("year", "size"),
    median_duration_min = ("duration_min", "median"),
    pct_traveling       = ("traveling", "mean"),
    mean_observers      = ("n_observers", "mean"),
    mean_species        = ("n_species", "mean"),
    localities          = ("locality_id", "nunique"),
)
effort["pct_traveling"] *= 100
effort.to_csv(f"{RESULTS}/rq3_effort_by_year.csv")

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, effort.columns):
    ax.plot(effort.index, effort[col], marker="o")
    ax.set_title(col.replace("_", " "))
    ax.grid(alpha=.3)
fig.suptitle("RQ3a. Birder effort on Dauphin Island in spring, by year", fontsize=13)
fig.tight_layout()
fig.savefig(f"{RESULTS}/rq3_effort_by_year.png", dpi=150)
effort

In [ ]:
# Where and when people bird: season, time of day, and the top localities
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
wide["day_of_year"].hist(bins=40, ax=axes[0]); axes[0].set_title("Checklists by day of year")
wide["start_hour"].hist(bins=24, ax=axes[1]);  axes[1].set_title("Checklists by start hour")
top = zf.drop_duplicates("checklist_id")["locality"].value_counts().head(10)[::-1]
top.plot.barh(ax=axes[2]); axes[2].set_title("Top 10 localities")
fig.tight_layout()
fig.savefig(f"{RESULTS}/rq3_effort_distribution.png", dpi=150)
print(f"Share of checklists at the top 3 localities: {top[::-1].head(3).sum() / len(wide):.0%}")

## RQ3b. Covariate importance: logistic regression

For each species, fit the full model, then refit with each covariate group removed. **ΔAIC** is how much worse the model gets without that group:

- **> 10**: the covariate clearly matters
- **2–10**: some support
- **< 2**: negligible

In [ ]:
FULL_RHS = f"year_c + {COVARS}"
daic = {}
for sp in SPECIES:
    d = species_df(wide, sp)
    full = fit_glm(d, FULL_RHS)
    if full is None:
        continue
    row = {}
    for name, term in {**COV_TERMS, "year": "year_c"}.items():
        reduced = fit_glm(d, FULL_RHS.replace(term, "").replace("+  +", "+").strip(" +"))
        row[name] = (reduced.aic - full.aic) if reduced is not None else np.nan
    daic[sp] = row

daic = pd.DataFrame(daic).T
daic.loc["MEAN (all species)"] = daic.mean()
daic = daic.round(1)
daic.to_csv(f"{RESULTS}/rq3_delta_aic.csv")
daic.style.background_gradient(cmap="Reds", axis=None, vmin=0, vmax=50).format("{:.1f}")

## RQ3c. Covariate importance: random forest

A random forest can find nonlinear effects and interactions that the regression misses. **Permutation importance** measures how much test-set AUC drops when one covariate's values are shuffled.

In [ ]:
RF_FEATURES = ["year", "day_of_year", "start_hour", "duration_min", "distance_km",
               "n_observers", "traveling", "other_species"]

def fit_rf(X, y):
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=10, max_features="sqrt",
                                  class_weight="balanced", n_jobs=-1, random_state=SEED).fit(X, y)

rf_imp = {}
for sp in SPECIES:
    d = species_df(wide, sp)
    if d["detected"].sum() < 30:
        continue
    X_tr, X_te, y_tr, y_te = train_test_split(d[RF_FEATURES], d["detected"], test_size=.3,
                                              stratify=d["detected"], random_state=SEED)
    rf = fit_rf(X_tr, y_tr)
    pi = permutation_importance(rf, X_te, y_te, scoring="roc_auc", n_repeats=5,
                                random_state=SEED, n_jobs=-1)
    rf_imp[sp] = pd.Series(pi.importances_mean, index=RF_FEATURES)

rf_imp = pd.DataFrame(rf_imp).T
rf_imp.loc["MEAN (all species)"] = rf_imp.mean()
rf_imp.to_csv(f"{RESULTS}/rq3_rf_importance.csv")

ax = rf_imp.loc["MEAN (all species)"].sort_values().plot.barh(figsize=(7, 4))
ax.set_title("RQ3c. Mean permutation importance (drop in AUC)")
plt.tight_layout(); plt.savefig(f"{RESULTS}/rq3_rf_importance.png", dpi=150)
rf_imp.style.background_gradient(cmap="Blues", axis=None).format("{:.3f}")

In [ ]:
# Shape of each effect for one species (partial dependence)
from sklearn.inspection import PartialDependenceDisplay
sp = MIGRANTS[0]                     # change to look at another species
d = species_df(wide, sp)
X = d[RF_FEATURES].astype(float)     # partial dependence needs float columns
rf = fit_rf(X, d["detected"])
fig, ax = plt.subplots(figsize=(14, 6))
PartialDependenceDisplay.from_estimator(rf, X, RF_FEATURES, n_cols=4,
                                        grid_resolution=30, ax=ax)
plt.suptitle(f"RQ3c. Partial dependence: {sp}"); plt.tight_layout()
plt.savefig(f"{RESULTS}/rq3_partial_dependence.png", dpi=150)

## RQ3d. Does heavy birding at hotspots distort the trend?

Compare each trend on **all checklists** with the trend after **spatial subsampling** (one checklist per grid cell per week, averaged over several draws). Large differences mean hotspot effort is pulling the estimate.

In [ ]:
rows = []
for sp in SPECIES:
    all_data = annual_trend(species_df(wide, sp))
    subs = [annual_trend(species_df(spatial_subsample(wide, SEED + i), sp)) for i in range(20)]
    rows.append({"species": sp, "all checklists": all_data, "spatially subsampled": np.nanmean(subs)})
spatial = pd.DataFrame(rows).set_index("species")
spatial["difference"] = spatial["spatially subsampled"] - spatial["all checklists"]
spatial.to_csv(f"{RESULTS}/rq3_spatial_subsampling.csv")
spatial

### Writing up RQ3
- **Bias over time:** use the RQ3a plots to show whether effort grew. Raw totals then reflect effort, not birds.
- **Most relevant covariates:** rank them by the "MEAN" rows of the ΔAIC and random-forest tables. Note where the two methods agree.
- **Season, time and place:** day of year (migration timing), start hour, and locality concentration (RQ3a, RQ3d).
- **Corrections adopted:** complete checklists + zero-filling, effort filters, effort covariates, standard-checklist predictions, spatial subsampling.

---
# RQ1: How are the migrant populations changing over time?

## RQ1a. Three versions of the trend

For each species and year:
1. **Raw detections**: the number of checklists reporting it. Expected to be biased by effort.
2. **Reporting rate**: detections ÷ checklists.
3. **Adjusted index**: modeled detection probability on the standard checklist (1 h, 1 km, 1 observer, 7 AM, peak date).

Each series is divided by its own mean so all three fit on one axis.

In [ ]:
n_cl = wide.groupby("year").size()
series = {}
for sp in SPECIES:
    raw  = wide.groupby("year")[sp].sum()
    rate = raw / n_cl
    adj  = yearly_index(species_df(wide, sp))
    series[sp] = pd.DataFrame({"raw detections": raw, "reporting rate": rate, "adjusted index": adj})

ncol = 3
nrow = int(np.ceil(len(SPECIES) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2 * nrow), sharex=True)
for ax, sp in zip(axes.flat, SPECIES):
    s = series[sp]
    for col, style in zip(s.columns, [":", "--", "-"]):
        ax.plot(s.index, s[col] / s[col].mean(), style, marker="o", ms=3, label=col)
    ax.set_title(sp + (" (control)" if sp in CONTROL_SPECIES else ""))
    ax.axhline(1, color="grey", lw=.5); ax.grid(alpha=.3)
for ax in axes.flat[len(SPECIES):]:
    ax.set_visible(False)
axes.flat[0].legend(fontsize=8)
fig.suptitle("RQ1a. Raw vs. reporting rate vs. effort-adjusted trend (each relative to its mean)", fontsize=13)
fig.tight_layout()
fig.savefig(f"{RESULTS}/rq1_trends_by_species.png", dpi=150)

pd.concat(series, names=["species", "year"]).to_csv(f"{RESULTS}/rq1_yearly_series.csv")

## RQ1b. Trend estimates with confidence intervals

`trend_%/yr` is the average yearly % change in detection probability on the standard checklist. The 95% CI comes from **bootstrapping**: each run spatially subsamples, resamples checklists within each year, and refits.

**This cell is the slowest one** (N_BOOT × species fits). Keep `N_BOOT = 100` while you're developing and raise it for the final report.

In [ ]:
trends = trend_table(wide, SPECIES)
trends["type"] = ["control" if s in CONTROL_SPECIES else "migrant" for s in trends.index]
trends.to_csv(f"{RESULTS}/rq1_trend_table.csv")

t = trends.sort_values("trend_%/yr")
fig, ax = plt.subplots(figsize=(8, 0.45 * len(t) + 1))
colors = ["grey" if ty == "control" else "C0" for ty in t["type"]]
for i, (sp, r) in enumerate(t.iterrows()):
    ax.errorbar(r["trend_%/yr"], i, xerr=[[r["trend_%/yr"] - r["ci_low"]], [r["ci_high"] - r["trend_%/yr"]]],
                fmt="o", color=colors[i], capsize=3)
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.index)
ax.axvline(0, color="k", lw=.8)
ax.set_xlabel("% change per year (95% CI)")
ax.set_title("RQ1b. Effort-adjusted spring trends, Dauphin Island (grey = controls)")
plt.tight_layout(); plt.savefig(f"{RESULTS}/rq1_trend_forest.png", dpi=150)
trends

## RQ1c. Migrant community trend

The **geometric mean** of the migrant species' adjusted indices, each scaled to its own mean. This is the same approach used for multi-species bird indicators. It answers "is the migrant community as a whole changing?"

In [ ]:
idx = pd.DataFrame({sp: series[sp]["adjusted index"] for sp in MIGRANTS})
idx = idx / idx.mean()
community = np.exp(np.log(idx.clip(lower=1e-6)).mean(axis=1))
slope, intercept, r, p, se = stats.linregress(community.index, np.log(community))
print(f"Community trend: {(np.exp(slope) - 1) * 100:+.2f}% per year (p = {p:.3f})")

ax = community.plot(marker="o", figsize=(8, 4))
ax.plot(community.index, np.exp(intercept + slope * community.index), "--", color="grey")
ax.axhline(1, color="k", lw=.5)
ax.set_title("RQ1c. Migrant community index (geometric mean of adjusted indices)")
ax.set_ylabel("relative to mean"); plt.tight_layout()
plt.savefig(f"{RESULTS}/rq1_community_index.png", dpi=150)
community.rename("community_index").to_csv(f"{RESULTS}/rq1_community_index.csv")

## RQ1d. Possible causes (optional: weather)

Dauphin Island counts depend heavily on **weather**. Fallouts happen when north winds and rain force trans-Gulf migrants down on the coast. A run of good- or bad-weather springs can look like a population trend.

To test that, download daily weather for a nearby station (e.g., NOAA daily data for Mobile, AL). Save it as `weather.csv` with columns `obs_date, wind_dir, wind_speed, precip`, then run the next cell. **If the trend shrinks once weather is in the model, part of the apparent trend was weather.**

Any other cause (habitat change on the island, declines on the wintering grounds) can only be raised as a **hypothesis**. Compare your species list against what the literature reports.

In [ ]:
if os.path.exists(WEATHER_CSV):
    wx = pd.read_csv(WEATHER_CSV, parse_dates=["obs_date"])
    wx["north_wind"] = np.cos(np.radians(wx["wind_dir"])) * wx["wind_speed"]   # + = from the north
    wx["rain"] = (wx["precip"] > 0).astype(int)
    ww = wide.reset_index().merge(wx[["obs_date", "north_wind", "rain"]], on="obs_date", how="left") \
             .set_index("checklist_id").dropna(subset=["north_wind"])
    rows = []
    for sp in SPECIES:
        d = species_df(ww, sp)
        base = fit_glm(d, f"year_c + {COVARS}")
        wxm  = fit_glm(d, f"year_c + {COVARS} + north_wind + rain")
        rows.append({"species": sp,
                     "year coef (no weather)": base.params["year_c"] if base is not None else np.nan,
                     "year coef (with weather)": wxm.params["year_c"] if wxm is not None else np.nan,
                     "north wind coef": wxm.params["north_wind"] if wxm is not None else np.nan})
    weather_tbl = pd.DataFrame(rows).set_index("species")
    weather_tbl.to_csv(f"{RESULTS}/rq1_weather.csv")
    display(weather_tbl)
else:
    print("No weather.csv found, skipping.")

---
# RQ2: How do we know the analysis is reliable?

Five independent checks. A trend that passes all of them is credible.

## RQ2a. Did the effort correction work?

If the correction worked, the **adjusted index should no longer rise and fall with the number of checklists**. Compare the correlation with yearly checklist count before and after adjustment.

In [ ]:
rows = []
for sp in SPECIES:
    s = series[sp].dropna()
    n = n_cl.loc[s.index]
    rows.append({"species": sp,
                 "r(raw detections, checklists)": stats.spearmanr(s["raw detections"], n)[0],
                 "r(reporting rate, checklists)": stats.spearmanr(s["reporting rate"], n)[0],
                 "r(adjusted index, checklists)": stats.spearmanr(s["adjusted index"], n)[0]})
effort_check = pd.DataFrame(rows).set_index("species")
effort_check.loc["MEAN"] = effort_check.mean()
effort_check.to_csv(f"{RESULTS}/rq2_effort_correlation.csv")
effort_check.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1, axis=None).format("{:.2f}")

## RQ2b. Prediction on unseen data

1. **Temporal holdout:** train on all years except the last `HOLDOUT_YEARS`, test on those.
2. **Spatial block cross-validation:** 5 folds, holding out whole grid cells at a time.

Metrics:
- **AUC**: 0.5 = no better than chance, 1 = perfect ranking.
- **Brier score**: lower is better. Compare it with "no-skill", which predicts the training detection rate for every checklist.

In [ ]:
cutoff = YEARS.max() - HOLDOUT_YEARS
rows = []
for sp in SPECIES:
    d = species_df(wide, sp)
    tr, te = d[d["year"] <= cutoff], d[d["year"] > cutoff]
    if tr["detected"].sum() < 20 or te["detected"].nunique() < 2:
        continue
    glm = fit_glm(tr, f"year_c + {COVARS}")
    p_glm = glm.predict(te) if glm is not None else np.full(len(te), np.nan)
    rf = fit_rf(tr[RF_FEATURES], tr["detected"])
    p_rf = rf.predict_proba(te[RF_FEATURES])[:, 1]

    # spatial block CV (GLM)
    gkf = GroupKFold(n_splits=5)
    p_cv = pd.Series(np.nan, index=d.index)
    for tr_i, te_i in gkf.split(d, groups=d["cell"]):
        f = fit_glm(d.iloc[tr_i], f"year_c + {COVARS}")
        if f is not None:
            p_cv.iloc[te_i] = f.predict(d.iloc[te_i]).values
    ok = p_cv.notna()

    rows.append({"species": sp,
                 "AUC GLM (holdout yrs)": roc_auc_score(te["detected"], p_glm),
                 "AUC RF (holdout yrs)":  roc_auc_score(te["detected"], p_rf),
                 "Brier GLM": brier_score_loss(te["detected"], p_glm),
                 "Brier RF":  brier_score_loss(te["detected"], p_rf),
                 "Brier no-skill": brier_score_loss(te["detected"], np.full(len(te), tr["detected"].mean())),
                 "AUC GLM (spatial CV)": roc_auc_score(d.loc[ok, "detected"], p_cv[ok])})
validation = pd.DataFrame(rows).set_index("species")
validation.loc["MEAN"] = validation.mean()
validation.to_csv(f"{RESULTS}/rq2_predictive_validation.csv")
validation

## RQ2c. Permutation test: could the trend be chance?

Shuffle the year labels among checklists, which destroys any real trend, and refit `N_PERM` times. The **p-value** is the share of shuffled trends at least as large as the real one.

In [ ]:
rows = []
for sp in SPECIES:
    d = species_df(wide, sp)
    obs = annual_trend(d)
    null = []
    for i in range(N_PERM):
        dp = d.copy()
        perm = rng.permutation(len(dp))
        dp["year"] = dp["year"].values[perm]
        dp["year_c"] = dp["year_c"].values[perm]
        null.append(annual_trend(dp))
    null = np.array(null)
    null = null[np.isfinite(null)]
    rows.append({"species": sp, "observed %/yr": obs,
                 "null 95% range": f"{np.percentile(null, 2.5):+.2f} to {np.percentile(null, 97.5):+.2f}",
                 "p-value": (np.sum(np.abs(null) >= abs(obs)) + 1) / (len(null) + 1)})
perm_tbl = pd.DataFrame(rows).set_index("species")
perm_tbl.to_csv(f"{RESULTS}/rq2_permutation_test.csv")
perm_tbl

## RQ2d. Robustness: does the trend survive different choices?

Re-estimate each trend under **stricter filters** and **without spatial subsampling**. (The dataset notebook already applied the loosest filters, ≤5 h and ≤5 km, so only stricter versions are possible here. To test looser ones, rerun that notebook.)

The last column counts how many versions agree in direction with the main estimate.

In [ ]:
variants = {
    "main (all data)":        wide,
    "duration <= 3 h":        wide[wide["duration_min"] <= 180],
    "distance <= 2 km":       wide[wide["distance_km"] <= 2],
    "observers <= 4":         wide[wide["n_observers"] <= 4],
    "traveling only":         wide[wide["traveling"] == 1],
    "stationary only":        wide[wide["traveling"] == 0],
    "drop first 3 years":     wide[wide["year"] > YEARS.min() + 2],
    "spatial subsample":      spatial_subsample(wide, SEED),
}
sens = pd.DataFrame({name: {sp: annual_trend(species_df(w, sp)) for sp in SPECIES}
                     for name, w in variants.items()})
main_sign = np.sign(sens["main (all data)"])
sens["same direction"] = (np.sign(sens.drop(columns="main (all data)")).eq(main_sign, axis=0)).sum(axis=1) \
                         .astype(str) + f"/{len(variants) - 1}"
sens.to_csv(f"{RESULTS}/rq2_sensitivity.csv")
sens

## RQ2e. Resident control species

Residents like Northern Cardinal and Carolina Wren aren't migrating across the Gulf, so a large trend in them may point to **leftover bias** in the method. Residents can have real trends too, though, so treat this as a warning sign rather than proof.

In [ ]:
trends.loc[trends["type"] == "control", ["trend_%/yr", "ci_low", "ci_high", "direction"]]

## RQ2f. Agreement with independent estimates

Make a CSV called `external_trends.csv` with columns:

| species | external_trend | source |
|---|---|---|
| Wood Thrush | -1.8 | BBS Southeast 2008–2022 |

- **BBS:** USGS North American Breeding Bird Survey trend estimates (annual % change) for the species, by region or survey-wide.
- **eBird Status & Trends:** trend estimates from the Cornell Lab (separate access request).

Use the same years as your data as far as possible. Running the next cell without the file creates a template.

In [ ]:
if not os.path.exists(EXTERNAL_TRENDS):
    pd.DataFrame({"species": MIGRANTS, "external_trend": np.nan, "source": ""}) \
      .to_csv(EXTERNAL_TRENDS, index=False)
    print(f"Created template {EXTERNAL_TRENDS}. Fill in the external_trend column and rerun.")
else:
    ext = pd.read_csv(EXTERNAL_TRENDS).dropna(subset=["external_trend"]).set_index("species")
    comp = trends[["trend_%/yr", "ci_low", "ci_high"]].join(ext, how="inner")
    if len(comp) >= 3:
        agree = (np.sign(comp["trend_%/yr"]) == np.sign(comp["external_trend"])).mean()
        rho, p = stats.spearmanr(comp["trend_%/yr"], comp["external_trend"])
        print(f"Direction agrees for {agree:.0%} of {len(comp)} species; Spearman rho = {rho:.2f} (p = {p:.3f})")
        ax = comp.plot.scatter("external_trend", "trend_%/yr", figsize=(6, 6))
        for sp, r in comp.iterrows():
            ax.annotate(sp, (r["external_trend"], r["trend_%/yr"]), fontsize=7)
        lim = np.abs(comp[["external_trend", "trend_%/yr"]].values).max() * 1.2
        ax.plot([-lim, lim], [-lim, lim], "--", color="grey"); ax.axhline(0, lw=.5); ax.axvline(0, lw=.5)
        ax.set_xlabel("External trend (%/yr)"); ax.set_ylabel("Dauphin Island trend (%/yr)")
        ax.set_title("RQ2f. Our trends vs. independent estimates")
        plt.tight_layout(); plt.savefig(f"{RESULTS}/rq2_external_comparison.png", dpi=150)
        comp.to_csv(f"{RESULTS}/rq2_external_comparison.csv")
        display(comp)
    else:
        print("Fill in at least 3 species in", EXTERNAL_TRENDS)

### Writing up RQ2: summary table

In [ ]:
summary = trends[["trend_%/yr", "ci_low", "ci_high", "direction"]].join(
    perm_tbl[["p-value"]]).join(sens[["same direction"]]).join(
    effort_check[["r(raw detections, checklists)", "r(adjusted index, checklists)"]])
summary.to_csv(f"{RESULTS}/rq2_summary.csv")
summary

---
# RQ4: Does a northern site support the trends?

1. Pick a northern spring migration hotspot on the same flyway that the same species pass through.
2. Make a copy of **`dauphin_dataset.ipynb`** and change its Settings:
   - `SRC_DB`: an eBird download covering that site
   - the bounding box
   - `SPRING_START` / `SPRING_END`, shifted to that site's peak (later in spring)
   - `OUT_DB = "north.duckdb"`, `OUT_DIR = "data_north"`
3. Run it, then run the cell below.

**Interpretation:**
- **Same direction at both sites:** supports a real population change.
- **Disagreement:** points to local factors (e.g., habitat change on Dauphin Island) or stopover behavior. In good weather many migrants fly over the coast without landing.

In [ ]:
if os.path.exists(NORTH_DATA):
    wide_n = make_wide(pd.read_parquet(NORTH_DATA))
    common = [s for s in MIGRANTS if s in wide_n.columns and wide_n[s].sum() >= 20]
    trends_n = trend_table(wide_n, common)
    trends_n.to_csv(f"{RESULTS}/rq4_north_trend_table.csv")

    both = trends.loc[common, ["trend_%/yr", "ci_low", "ci_high"]].join(
        trends_n[["trend_%/yr", "ci_low", "ci_high"]], lsuffix=" Dauphin", rsuffix=" north")
    agree = (np.sign(both["trend_%/yr Dauphin"]) == np.sign(both["trend_%/yr north"])).mean()
    rho, p = stats.spearmanr(both["trend_%/yr Dauphin"], both["trend_%/yr north"])
    print(f"Same direction for {agree:.0%} of {len(both)} species; Spearman rho = {rho:.2f} (p = {p:.3f})")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.errorbar(both["trend_%/yr north"], both["trend_%/yr Dauphin"],
                xerr=[both["trend_%/yr north"] - both["ci_low north"], both["ci_high north"] - both["trend_%/yr north"]],
                yerr=[both["trend_%/yr Dauphin"] - both["ci_low Dauphin"], both["ci_high Dauphin"] - both["trend_%/yr Dauphin"]],
                fmt="o", capsize=2, alpha=.7)
    for sp, r in both.iterrows():
        ax.annotate(sp, (r["trend_%/yr north"], r["trend_%/yr Dauphin"]), fontsize=7)
    lim = np.nanmax(np.abs(both.values)) * 1.1
    ax.plot([-lim, lim], [-lim, lim], "--", color="grey"); ax.axhline(0, lw=.5); ax.axvline(0, lw=.5)
    ax.set_xlabel("Northern site trend (%/yr)"); ax.set_ylabel("Dauphin Island trend (%/yr)")
    ax.set_title("RQ4. Trend agreement between sites")
    plt.tight_layout(); plt.savefig(f"{RESULTS}/rq4_site_comparison.png", dpi=150)
    both.to_csv(f"{RESULTS}/rq4_site_comparison.csv")
    display(both)
else:
    print(f"No {NORTH_DATA} yet. Run the dataset notebook for the northern site first.")

---
# Outputs for the report

Everything is in `results/`:

| RQ | Files |
|---|---|
| RQ3 | `rq3_effort_by_year.*`, `rq3_effort_distribution.png`, `rq3_delta_aic.csv`, `rq3_rf_importance.*`, `rq3_partial_dependence.png`, `rq3_spatial_subsampling.csv` |
| RQ1 | `rq1_trends_by_species.png`, `rq1_trend_table.csv`, `rq1_trend_forest.png`, `rq1_community_index.*`, `rq1_weather.csv` |
| RQ2 | `rq2_effort_correlation.csv`, `rq2_predictive_validation.csv`, `rq2_permutation_test.csv`, `rq2_sensitivity.csv`, `rq2_external_comparison.*`, `rq2_summary.csv` |
| RQ4 | `rq4_north_trend_table.csv`, `rq4_site_comparison.*` |

In [ ]:
sorted(os.listdir(RESULTS))